In [1]:
# 1. Install system dependency for PDF processing
!apt-get update
!apt-get install poppler-utils -y

# 2. Install Python dependencies (ignoring dependency resolver warnings if they pop up)
!pip install paddlepaddle paddleocr pdf2image pillow langchain langchain-community

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,786 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get

In [2]:
import os
import sys
import types

# Bypass the slow model hoster check
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# Patch the LangChain bug in memory
m1 = types.ModuleType("langchain.docstore.document")
m1.Document = object
sys.modules["langchain.docstore.document"] = m1

m2 = types.ModuleType("langchain.text_splitter")
m2.RecursiveCharacterTextSplitter = object
sys.modules["langchain.text_splitter"] = m2

# Safely import and initialize
from pdf2image import convert_from_path
from paddleocr import PaddleOCR

# Initialize PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='en')
print("PaddleOCR Initialized Successfully!")

Connectivity check to the model hoster has been skipped because `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` is enabled.
/tmp/ipykernel_3331/2765804581.py:22: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='en')
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN`

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


...

Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('en_PP-OCRv5_mobile_rec', None)
Using official model (en_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

PaddleOCR Initialized Successfully!


In [3]:
from PIL import Image
import os

def extract_bill_text(file_path):
    extracted_text = []

    # 1. Handle PDFs by converting them to images first
    if file_path.lower().endswith('.pdf'):
        print("Converting PDF to images...")
        images = convert_from_path(file_path, dpi=300)
        for i, image in enumerate(images):
            img_path = f"temp_page_{i}.jpg"
            image.save(img_path, 'JPEG')

            print(f"Running OCR on page {i+1}...")
            # Run PaddleOCR on the temporary image
            result = ocr.ocr(img_path, cls=True)
            extracted_text.append(parse_ocr_result(result))

            # Clean up temp file
            if os.path.exists(img_path):
                os.remove(img_path)

    # 2. Handle standard Images
    elif file_path.lower().endswith(('.png', '.jpg', '.jpeg')):
        print("Running OCR on image...")
        result = ocr.ocr(file_path, cls=True)
        extracted_text.append(parse_ocr_result(result))

    else:
        return "Unsupported file format. Please provide PDF or Image."

    return "\n".join(extracted_text)

def parse_ocr_result(result):
    page_text = []
    # PaddleOCR's latest output format can sometimes be a list of lists depending on the backend
    if not result or result[0] is None:
        return ""

    for res in result:
        for line in res:
            # line format is generally: [[x,y coords], [text, confidence]]
            text = line[1][0]
            confidence = line[1][1]
            if confidence > 0.70: # Filter out low-confidence scribbles
                page_text.append(text)

    return " ".join(page_text)

In [1]:
!pip install google-genai google-auth-httplib2

  Obtaining dependency information for google-auth-httplib2 from https://files.pythonhosted.org/packages/97/e9/93afb14d23a949acaa3f4e7cc51a0024671174e116e35f42850764b99634/google_auth_httplib2-0.3.1-py3-none-any.whl.metadata
  Obtaining dependency information for httplib2<1.0.0,>=0.19.0 from https://files.pythonhosted.org/packages/2f/90/fd509079dfcab01102c0fdd87f3a9506894bc70afcf9e9785ef6b2b3aff6/httplib2-0.31.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/91.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/91.1 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/91.1 kB ? eta -:--:--
   ------------- -------------------------- 30.7/91.1 kB 435.7 kB/s eta 0:00:01
   ----------------- ---------------------- 41.0/91.1 kB 393.8 kB/s eta 0:00:01
   ---------------------------------------- 91.1/91.1 kB 573.5 kB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
from google import genai

# Initialize the LLM client with API key
# IMPORTANT: Set your Gemini API key before running this cell
# Get it from: https://ai.google.dev/
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

if GEMINI_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️  WARNING: GEMINI_API_KEY not set. Structure extraction will be disabled.")
    print("Please set: export GEMINI_API_KEY='your-key-here'")
    client = None
else:
    genai.configure(api_key=GEMINI_API_KEY)
    client = genai.Client()
    print("✅ Gemini API initialized successfully!")

def structure_pharmacy_invoice(raw_text):
    """
    Extract structured data from pharmacy/supplier invoice using Google Gemini.
    Returns JSON with medicine details, quantities, rates, and totals.
    """
    if not client:
        return {"error": "Gemini API key not configured. Please set GEMINI_API_KEY environment variable."}
    
    prompt = f"""
    You are a highly accurate pharmacy invoice extraction system.
    Your task is to extract key entities from the raw OCR text of a pharmacy supplier invoice.

    RAW OCR TEXT:
    {raw_text}

    INSTRUCTIONS:
    1. Extract the supplier's name and address.
    2. Extract the invoice number and date (Format: YYYY-MM-DD).
    3. Extract all medicine entries with the following details for each:
       - Medicine name/brand
       - Generic composition (if available)
       - Quantity/Batch quantity
       - Unit rate (price per unit)
       - HSN/SAC code (for GST calculation)
       - Batch number (if visible)
       - Expiry date (if visible, Format: YYYY-MM-DD)
    4. Extract the subtotal, GST amount, and total amount.
    5. If a value is missing or illegible, return null. Do not guess.

    Return the extracted data STRICTLY as a valid JSON object with this structure:
    {{
        "supplier": {{"name": "", "address": ""}},
        "invoice_number": "",
        "invoice_date": "",
        "medicines": [
            {{
                "name": "",
                "composition": "",
                "quantity": "",
                "unit_rate": "",
                "hsn_code": "",
                "batch_number": "",
                "expiry_date": ""
            }}
        ],
        "subtotal": "",
        "gst_amount": "",
        "total_amount": ""
    }}
    """

    try:
        print("📄 Structuring invoice data with Gemini LLM...")
        response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompt,
        )
        
        # Parse the JSON response
        response_text = response.text
        # Try to extract JSON if there's extra text
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()
        
        structured_data = json.loads(response_text)
        return structured_data
    except json.JSONDecodeError:
        print("❌ Failed to parse LLM response as JSON")
        return {"error": "LLM response was not valid JSON", "raw_response": response.text}
    except Exception as e:
        print(f"❌ Error during structure extraction: {str(e)}")
        return {"error": str(e)}

# --- COMPLETE WORKFLOW EXAMPLE ---
def process_pharmacy_invoice(file_path):
    """
    Complete end-to-end pipeline: Extract OCR text → Structure with LLM
    """
    print(f"\n🔄 Processing invoice: {file_path}")
    
    # Step 1: Extract raw OCR text
    raw_text = extract_bill_text(file_path)
    if isinstance(raw_text, str) and raw_text.startswith("Unsupported"):
        return {"error": raw_text}
    
    print(f"\n📝 Extracted Raw Text:\n{raw_text[:500]}...")
    
    # Step 2: Structure the data
    structured_data = structure_pharmacy_invoice(raw_text)
    print(f"\n✅ Structured Data:\n{json.dumps(structured_data, indent=2)}")
    
    return structured_data

# --- HOW TO USE IT ---
# 1. Upload your pharmacy invoice to Colab (e.g., "sample_invoice.jpg")
# 2. Set your API key (in Colab secrets or environment):
#    os.environ["GEMINI_API_KEY"] = "your-api-key"
# 3. Run the complete workflow:
#    result = process_pharmacy_invoice("sample_invoice.jpg")
#    print(json.dumps(result, indent=2))

In [ ]:
# Test helper function (for debugging)
def test_ocr_pipeline():
    """
    Test the OCR pipeline with mock data (useful for debugging)
    """
    mock_invoice_text = """
    SUNDAR PHARMACY SUPPLIERS PVT LTD
    Invoice # INV-2024-001234
    Date: 2024-01-15
    
    Paracetamol 500mg Tablet (Crocin) - 10 strips - Rs. 45.00 - Batch: B001 - Exp: 2025-06-30
    Ibuprofen 400mg (Brufen) - 5 blisters - Rs. 120.00 - Batch: B002 - Exp: 2025-08-15
    Amoxicillin 500mg (Amoxil) - 3 boxes - Rs. 250.00 - Batch: B003 - Exp: 2025-12-31
    
    Subtotal: Rs. 2,500.00
    GST (5%): Rs. 125.00
    Total: Rs. 2,625.00
    """
    
    print("🧪 Testing with mock invoice text...")
    result = structure_pharmacy_invoice(mock_invoice_text)
    print("\n✅ Mock Test Result:")
    print(json.dumps(result, indent=2))

# Uncomment to run test:
# test_ocr_pipeline()